# Креды

In [1]:
# postgres_dev
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
credentials

{'host': '10.6.81.133',
 'port': '5432',
 'user': 'postgres',
 'password': 'tpY7H&sdvsdfsdf7zx9J'}

# Подключение

In [2]:
import geopandas as gpd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
# Подключение к БД
# engine = create_engine(f'postgresql://{credentials.get("user")}:pass@{credentials.get("host")}:{credentials.get("port")}/{credentials.get("password")}')

# Создаем URL через объект (автоматически экранирует)
url = URL.create(
    drivername='postgresql',
    username='postgres',
    password=credentials.get('password'),
    host=credentials.get("host"),
    port=credentials.get("port"),
    database='gisdb_8411_250226',
      query={
        'options': '-c search_path=egip,public'
    }
)

engine = create_engine(url)

# Сохраняем в файл

In [18]:


sql = """SELECT 
    id, 
    layer_id, 
    geometry
FROM (
    SELECT 
        id, 
        layer_id, 
        ST_Transform(
            ST_SetSRID(ST_GeomFromWKB(geometry), 4326), 
            4326
        ) AS geometry,
        ST_GeoHash(
            ST_Transform(
                ST_SetSRID(ST_GeomFromWKB(centroid), 4326), 
                4326
            )
        ) AS geohash
    FROM public.features_plain
    WHERE layer_alias IS NULL
) AS subquery

ORDER BY geohash
LIMIT 1; """

In [19]:
# Читаем в GeoDataFrame, указываем основную геометрию
gdf = gpd.GeoDataFrame.from_postgis(sql, engine, geom_col='geometry', crs='epsg:4326')
gdf.to_parquet(r'D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_1.geoparquet', engine='pyarrow')

# Читаем файл

In [20]:
# Загружаем ваш GeoParquet файл
gdf = gpd.read_parquet(r'D:\Artem\Work\amtech_projects\sedona_geoservice\data\features_1.geoparquet')
print(gdf.head())

        id  layer_id                                           geometry
0  3724141       170  MULTIPOLYGON (((36.99299 55.18414, 36.99288 55...


# Инициализируем Sedona

# Замеряем скорость выполнения вычислений

In [22]:
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнена за {end - start:.4f} секунд")
        return result
    return wrapper

In [ ]:
def sql_executor(query:str):
    return sedona.sql(query)

## Вычисление площади

In [23]:
sql_area = """ SELECT
 *,
 ST_Area(geaometry) AS area
 FROM features"""

# Обрабатываем features_1